In [ ]:
# transformers from Hugging Face is used to load and utilize pre-trained language models.
# accelerate is used for optimizing model loading and execution across different devices.
# torch is the underlying deep learning framework used by transformers.
!pip install -q gradio transformers accelerate torch

## Set up the Language Model

In [ ]:
import gradio as gr # to create the web-based chatbot interface.
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Define the model ID for the TinyLlama chatbot to specify which pre-trained model from Hugging Face will be used.
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the tokenizer for the specified model, responsible for converting text into numerical tokens that the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the pre-trained causal language model.
# AutoModelForCausalLM is used for models that predict the next token in a sequence.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto" # automatically assigns model layers to available devices (like CPU or GPU) for efficiency.
)

## Create a text-generation pipeline

In [ ]:
# This simplifies the process of using the model for text generation tasks.
pipe = pipeline(
    "text-generation",
    model=model,        # The loaded language model.
    tokenizer=tokenizer, # The loaded tokenizer.
    max_new_tokens=300,  # Maximum number of tokens to generate in the response.
    temperature=0.6,     # Controls the randomness of the generation. Lower values make output more deterministic.
    top_p=0.9,           # Filters out low probability words, improving diversity while maintaining coherence.
    repetition_penalty=1.1, # Penalizes repeated words to encourage more diverse output.
    do_sample=True,      # Enables sampling-based generation rather than greedy decoding.
    eos_token_id=tokenizer.eos_token_id, # End-of-sequence token ID to stop generation.
)

## Response function for the Gradio ChatInterface

In [ ]:
# This function takes user input (message) and chat history, then generates a chatbot response.
def respond(message, history):
    # Initialize the prompt with a system message, setting the chatbot's persona.
    prompt = "<|system|>\nYou are a helpful assistant.\n"

    # Iterate through the chat history to build the conversational context for the model.
    # Each turn consists of a user message and the bot's previous response.
    for user_msg, bot_msg in history:
        prompt += f"<|user|>\n{user_msg}\n<|assistant|>\n{bot_msg}\n"

    # Add the current user message to the prompt, preparing for the new assistant response.
    prompt += f"<|user|>\n{message}\n<|assistant|>\n"

    # Generate text using the Hugging Face pipeline.
    # The output is a list, and we extract the generated text from the first element.
    output = pipe(prompt)[0]["generated_text"]

    # Extract only the assistant's reply from the generated text.
    # The model might regenerate the prompt, so we split by the assistant token and take the last part.
    reply = output.split("<|assistant|>")[-1].strip()
    return reply

## Create and launch a Gradio ChatInterface.

In [ ]:
# This sets up the user interface for the chatbot.
demo = gr.ChatInterface(
    fn=respond, # The function that processes user input and generates responses.
    title="TinyLlama Chatbot", # Title displayed on the chatbot interface.
    description="A local Hugging Face chatbot running in Colab." # Description displayed below the title.
)

# share=True creates a public, shareable link (useful for Colab).
# debug=True provides more verbose output in the console, which can be helpful for debugging.
demo.launch(share=True, debug=True)